# CodonTransformer csi_top10_hc v2 fine-tuning

This entry introduces an E/K/L/S synonymous-family distribution regularizer while keeping masked-language modeling dominant. It reads train and validation only; the frozen test split is prohibited. Start with `RUN_MODE = "smoke"`. Change it to `"formal"` only after the eight-step smoke run has finite losses, changed model weights, a reloadable checkpoint, and verified translation.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
print(torch.cuda.get_device_name(0))

In [ ]:
import os
from pathlib import Path

RUN_MODE = os.environ.get("CODONTRANSFORMER_V2_RUN_MODE", "smoke")  # smoke or formal
assert RUN_MODE in {"smoke", "formal"}
REPO_URL = os.environ.get("GITHUB_REPO_URL", "https://github.com/Yuano0o/codontransformer-nb.git")
PROJECT_DIR = Path("/content/codontransformer-project")
UPSTREAM_REPO_URL = "https://github.com/Adibvafa/CodonTransformer.git"
UPSTREAM_COMMIT = "4a447b01dab860feb81b647ff1ff88ad598517f4"
HF_MODEL_ID = "adibvafa/CodonTransformer"
HF_MODEL_REVISION = "9744dcc920d813066391fc828d7a590207f148e8"
CONFIG_FILENAME = {
    "smoke": "smoke_test_cuda_csi_top10_v2.yaml",
    "formal": "finetune_cuda_csi_top10_v2.yaml",
}[RUN_MODE]
RUN_NAME = {
    "smoke": "finetune_csi_top10_hc_v2_smoke_v1",
    "formal": "finetune_csi_top10_hc_formal_v2",
}[RUN_MODE]
EXPECTED_COUNTS = {"train": 4524, "validation": 531}
EXPECTED_SHA256 = {
    "train": "bce089949fbbdee0c5e2403c22d6a1cba16a6cc851fdf731954af18f6b9df16d",
    "validation": "8e37ce0eff5684b1e42d6772fd3b0b6549a57d734fbb5e8fbc0ba4ec40058b49",
    "reference": "3d63f51d45972365fc92e7e20052ac82453cb10b09fcf26b76e5f6aaf0eec908",
}
AUTO_RESUME = True
print("RUN_MODE:", RUN_MODE, "CONFIG:", CONFIG_FILENAME)

## Securely clone the private repository
Store a read-only fine-grained token in the Colab secret `GITHUB_TOKEN`. The askpass helper prevents it from appearing in the command, output, exception, or notebook.

In [ ]:
import subprocess
import tempfile
from google.colab import userdata

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    raise RuntimeError("Colab secret GITHUB_TOKEN is unavailable") from None
if not github_token:
    raise RuntimeError("Colab secret GITHUB_TOKEN is empty")
with tempfile.TemporaryDirectory(prefix="git-askpass-") as askpass_dir:
    askpass_path = Path(askpass_dir) / "askpass.sh"
    askpass_path.write_text(
        '#!/bin/sh\ncase "$1" in\n  *sername*) printf \'%s\\n\' \'x-access-token\' ;;\n  *assword*) printf \'%s\\n\' "$COLAB_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n',
        encoding="utf-8",
    )
    askpass_path.chmod(0o700)
    clone_env = os.environ.copy()
    clone_env.update({"GIT_ASKPASS": str(askpass_path), "GIT_TERMINAL_PROMPT": "0", "COLAB_GITHUB_TOKEN": github_token})
    try:
        if not (PROJECT_DIR / ".git").is_dir():
            subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True, env=clone_env)
        else:
            subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True, env=clone_env)
    finally:
        clone_env.pop("COLAB_GITHUB_TOKEN", None)
        github_token = None
os.chdir(PROJECT_DIR)
UPSTREAM_DIR = PROJECT_DIR / "upstream"
if not (UPSTREAM_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", UPSTREAM_REPO_URL, str(UPSTREAM_DIR)], check=True)
subprocess.run(["git", "-C", str(UPSTREAM_DIR), "checkout", UPSTREAM_COMMIT], check=True)
print("Project:", PROJECT_DIR)

In [ ]:
subprocess.run([os.sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
upstream_pythonpath = str(UPSTREAM_DIR.resolve())
os.environ["PYTHONPATH"] = os.pathsep.join(value for value in (upstream_pythonpath, os.environ.get("PYTHONPATH", "")) if value)
if upstream_pythonpath not in os.sys.path:
    os.sys.path.insert(0, upstream_pythonpath)
import CodonTransformer
assert Path(CodonTransformer.__file__).resolve().is_relative_to(UPSTREAM_DIR.resolve())

## Mount Drive and freeze train/validation inputs
No test path is defined or opened anywhere in this notebook. v1 and smoke outputs use different directories and are never overwritten.

In [ ]:
import hashlib
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CodonTransformer")
DATA_BASE = DRIVE_ROOT / "data/stage2/final_csi_cohorts/experiments/csi_top10_hc"
TRAIN_PATH = DATA_BASE / "train.jsonl"
VALIDATION_PATH = DATA_BASE / "validation.jsonl"
REFERENCE_PATH = DRIVE_ROOT / "data/stage2/codon_reference.json"
RUN_DIR = DRIVE_ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()
def nonempty_lines(path):
    assert path.is_file(), f"Missing required input: {path}"
    with path.open(encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())
for split, path in (("train", TRAIN_PATH), ("validation", VALIDATION_PATH)):
    assert nonempty_lines(path) == EXPECTED_COUNTS[split]
    assert sha256(path) == EXPECTED_SHA256[split]
assert REFERENCE_PATH.is_file() and sha256(REFERENCE_PATH) == EXPECTED_SHA256["reference"]
print("Persistent v2 run:", RUN_DIR)

In [ ]:
MODEL_DIR = PROJECT_DIR / "models/pretrained"
subprocess.run([
    os.sys.executable, "scripts/download_pretrained.py",
    "--repo-id", HF_MODEL_ID,
    "--revision", HF_MODEL_REVISION,
    "--output-dir", str(MODEL_DIR),
], check=True)
assert (MODEL_DIR / "model.safetensors").is_file()

## Run v2 smoke or formal training
The formal run starts from the same official pretrained snapshot as v1. Automatic resume only inspects this v2 run's own Drive `last.ckpt`. Formal recovery replaces one `last.ckpt` every 200 optimizer steps.

In [ ]:
CONFIG_PATH = PROJECT_DIR / "configs" / CONFIG_FILENAME
subprocess.run([os.sys.executable, "scripts/finetune_codontransformer_v2.py", "--config", str(CONFIG_PATH), "--check-config"], check=True)
LAST_CHECKPOINT = RUN_DIR / "checkpoints/last.ckpt"
RESUME_CHECKPOINT = LAST_CHECKPOINT if AUTO_RESUME and LAST_CHECKPOINT.is_file() else None
command = [
    os.sys.executable, "scripts/finetune_codontransformer_v2.py",
    "--config", str(CONFIG_PATH),
    "--model-dir", str(MODEL_DIR),
    "--train-dataset-path", str(TRAIN_PATH),
    "--validation-dataset-path", str(VALIDATION_PATH),
    "--reference-json", str(REFERENCE_PATH),
    "--output-dir", str(RUN_DIR),
]
if RESUME_CHECKPOINT is not None:
    command.extend(["--resume-from-checkpoint", str(RESUME_CHECKPOINT)])
print("Resume:", RESUME_CHECKPOINT or "none; official pretrained initialization")
subprocess.run(command, check=True)

## Verify finite objectives, changed weights, checkpoint reload, and translation
This is a training-integrity check on the selected checkpoint, not biological model selection. Biological selection remains validation-only after a successful formal run.

In [ ]:
import json
import math
import pandas as pd
from safetensors import safe_open
from scripts.validate_checkpoint_inference import load_checkpoint

result = json.loads((RUN_DIR / "training_result.json").read_text(encoding="utf-8"))
BEST_CHECKPOINT = Path(result["best_checkpoint"])
LAST_CHECKPOINT = Path(result["last_checkpoint"])
assert BEST_CHECKPOINT.is_file() and LAST_CHECKPOINT.is_file()
metrics_path = max((RUN_DIR / "logs/lightning").glob("version_*/metrics.csv"), key=lambda path: path.stat().st_mtime)
metrics = pd.read_csv(metrics_path)
loss_columns = [column for column in metrics if "loss" in column]
assert any("mlm_loss" in column for column in loss_columns)
assert any("synonymous_jsd_loss" in column for column in loss_columns)
for column in loss_columns:
    values = metrics[column].dropna().tolist()
    assert values and all(math.isfinite(float(value)) for value in values), column
checkpoint_state = load_checkpoint(BEST_CHECKPOINT)
with safe_open(MODEL_DIR / "model.safetensors", framework="pt", device="cpu") as pretrained:
    comparable = [key for key in checkpoint_state if key in pretrained.keys()]
    assert comparable
    parameter_changed = any(not torch.equal(checkpoint_state[key], pretrained.get_tensor(key)) for key in comparable)
assert parameter_changed, "No model parameter changed from the official pretrained snapshot"
del checkpoint_state
INFERENCE_REPORT = RUN_DIR / "best_checkpoint_inference_validation.json"
subprocess.run([
    os.sys.executable, "scripts/validate_checkpoint_inference.py",
    "--model-dir", str(MODEL_DIR),
    "--checkpoint", str(BEST_CHECKPOINT),
    "--output", str(INFERENCE_REPORT),
    "--device", "cuda",
    "--organism", "Nicotiana tabacum",
], check=True)
inference = json.loads(INFERENCE_REPORT.read_text(encoding="utf-8"))
assert inference["translation_verified"] is True
{"run_mode": RUN_MODE, "finite_loss_columns": loss_columns, "parameter_changed": parameter_changed, "translation_verified": True}

In [ ]:
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN_DIR), path.stat().st_size)